# 0415 · LightGBM (ensamble)

**Qué hace.** Entrena y valida solo LightGBM, en las dos tareas:

| Tarea | Qué predice | Métrica para elegir |
| --- | --- | --- |
| `clf` | Probabilidad de incumplir el SLA | AP (mayor es mejor) |
| `reg` | Horas de última milla | MAE en horas (menor es mejor) |

**LightGBM** también es *boosting* de árboles, pero los hace crecer por hojas y agrupa los valores en histogramas, así que suele ser el más rápido con muchas guías. En regresión la búsqueda también elige la función de pérdida (`objetivo_reg`).

**Pasos.** 1) carga lo que dejó 0411; 2) búsqueda aleatoria de hiperparámetros con validación de origen móvil; 3) validación con la mejor configuración en los tres bloques (abril, mayo, junio); 4) vista rápida de sus métricas; 5) guarda predicciones, importancias y parámetros para 0416.

**No depende de los otros cuadernos de modelos.** Solo necesita que 0411 se haya corrido. La prueba final no se toca aquí.

**Tiempo.** La búsqueda hace `N_CONFIGS_BUSQUEDA × 3 bloques × 2 tareas` ajustes sobre una submuestra de `MAX_FILAS_BUSQUEDA` filas. Para una primera corrida rápida, bajar `N_CONFIGS_BUSQUEDA` a 3.

In [0]:
# Si faltan librerías:
# !pip install duckdb lightgbm xgboost scikit-learn joblib

In [0]:
%pip install "duckdb<2" lightgbm xgboost shap
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 93.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 125.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 142.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.3/547.3 kB 13.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.1/305.1 MB 116.1 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [shap]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 0. Configuración

In [0]:
import os, json, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore", category=UserWarning)

# MODULOS/ debe estar junto al cuaderno, como en 0100 (MODULOS/conexion_sql.py)
from MODULOS.modelado_temporal import (cargar_preparacion, pliegues_origen_movil, partes_del_bloque, marco_predicciones,
                                       construir_modelo, predecir, muestrear_configs, puntaje_config, mejores, params_de,
                                       importancia_pliegue, tabla_metricas)

In [0]:
MODELO = "LightGBM"
PATH_DATABRICKS = "/Volumes/proyectogrado/source/source/"
CARPETA_SALIDAS  = f"{PATH_DATABRICKS}LOG_CONSOLIDADO/REVISION_2026_10/V3/"
CARPETA_MODELADO = f"{CARPETA_SALIDAS}MODELADO/"

N_CONFIGS_BUSQUEDA = 20        # configuraciones aleatorias por tarea
MAX_FILAS_BUSQUEDA = 300_000   # submuestra de entrenamiento en la búsqueda (no en la validación)
EJECUTAR_BUSQUEDA  = True      # False: usa los parámetros que ya guardó este cuaderno
N_PERMUTACION      = 10_000    # filas de validación para la importancia por permutación en cada bloque

In [0]:
# Hiperparámetros de base (si no hay búsqueda) y espacio de búsqueda registrado (realimentación P4)
PARAMS_BASE = dict(n_estimators=400, learning_rate=0.05, num_leaves=63, subsample=0.8, subsample_freq=1,
                   colsample_bytree=0.8)
ESPACIO = dict(n_estimators=[200, 400, 800], learning_rate=[0.02, 0.05, 0.1], num_leaves=[31, 63, 127, 255],
               min_child_samples=[20, 50, 100, 200], subsample=[0.6, 0.8, 1.0], colsample_bytree=[0.6, 0.8, 1.0],
               reg_lambda=[0.0, 1.0, 5.0], objetivo_reg=["regression", "regression_l1", "huber"])

## 1. Cargar la preparación de 0411

Base de modelado, historia sin futuro de cada bloque y configuración común (periodos, semilla, variables). Los pliegues se vuelven a armar con la misma función del asesor.

In [0]:
CFG, df, HIST = cargar_preparacion(CARPETA_MODELADO)
SEMILLA, CATEGORICAS, VERSIONES, TAREAS = CFG["SEMILLA"], CFG["CATEGORICAS"], CFG["VERSIONES"], CFG["TAREAS"]
PLIEGUES = {t: list(pliegues_origen_movil(df, CFG["INICIOS_VALIDACION"], CFG["FIN_DESARROLLO"], t_etiqueta=te))
            for t, _, te in TAREAS}
print(f"{len(df):,} guías · versiones: {list(VERSIONES)} · preparado el {CFG['fecha_preparacion']}")

1,588,872 guías · versiones: ['internas', 'internas+clima'] · preparado el 2026-10-25T02:40:01.121225


## 2. Búsqueda aleatoria de hiperparámetros (solo en validación)

Se prueban `N_CONFIGS_BUSQUEDA` configuraciones de `ESPACIO`. Cada una se entrena en cada bloque con una submuestra de hasta `MAX_FILAS_BUSQUEDA` filas y se evalúa en el bloque completo. Se elige la de mejor promedio entre bloques. Si la base trae la versión «internas+clima_reajustada», se busca también para ella.

In [0]:
def buscar(version, numericas):
    """Prueba todas las configuraciones de una versión de variables; una fila por configuración."""
    filas = []
    for tarea, y_col, _ in TAREAS:
        for k, cfg in enumerate(muestrear_configs(ESPACIO, N_CONFIGS_BUSQUEDA, SEMILLA + len(MODELO))):
            if tarea == "clf":
                cfg = {a: b for a, b in cfg.items() if a != "objetivo_reg"}   # solo aplica a regresión
            vals = puntaje_config(tarea, MODELO, cfg, k, PLIEGUES[tarea], HIST, df, y_col,
                                  CATEGORICAS, numericas, MAX_FILAS_BUSQUEDA, SEMILLA)
            filas.append({"version": version, "tarea": tarea, "modelo": MODELO, "config": k,
                          "metrica": "AP" if tarea == "clf" else "MAE_h", "promedio": np.mean(vals),
                          "por_bloque": [round(v, 4) for v in vals], **{f"p_{a}": b for a, b in cfg.items()}})
            print(f"{version} {tarea} config {k}: {np.mean(vals):.4f}")
    return pd.DataFrame(filas)

In [0]:
RUTA_PARAMS = f"{CARPETA_MODELADO}parametros_{MODELO}.json"
t_ini = time.time()
if EJECUTAR_BUSQUEDA:
    versiones_busqueda = ["internas"] + [v for v in VERSIONES if v == "internas+clima_reajustada"]
    tabla_busqueda = pd.concat([buscar(v, VERSIONES[v]) for v in versiones_busqueda], ignore_index=True)
    tabla_busqueda.to_pickle(f"{CARPETA_MODELADO}busqueda_{MODELO}.pkl")       # exacta, para 0416
    tabla_busqueda.to_csv(f"{CARPETA_MODELADO}busqueda_{MODELO}.csv", index=False)
    MEJORES = mejores(tabla_busqueda, {MODELO: ESPACIO})
else:
    MEJORES = json.load(open(RUTA_PARAMS))["mejores"]
print(f"Búsqueda: {time.time() - t_ini:.0f} s")
display(pd.DataFrame(MEJORES).T)

internas clf config 0: 0.3085
internas clf config 1: 0.3322
internas clf config 2: 0.3168
internas clf config 3: 0.3432
internas clf config 4: 0.3390
internas clf config 5: 0.3207
internas clf config 6: 0.3169
internas clf config 7: 0.3279
internas clf config 8: 0.3318
internas clf config 9: 0.3205
internas clf config 10: 0.3335
internas clf config 11: 0.3151
internas clf config 12: 0.3247
internas clf config 13: 0.3240
internas clf config 14: 0.3547
internas clf config 15: 0.3192
internas clf config 16: 0.3251
internas clf config 17: 0.3371
internas clf config 18: 0.3470
internas clf config 19: 0.3552
internas reg config 0: 6.8905
internas reg config 1: 6.9368
internas reg config 2: 6.8733
internas reg config 3: 7.4286
internas reg config 4: 6.8733
internas reg config 5: 7.4846
internas reg config 6: 7.3662
internas reg config 7: 7.7940
internas reg config 8: 7.4585
internas reg config 9: 6.8670
internas reg config 10: 6.8783
internas reg config 11: 6.9292
internas reg config 12: 7.46

n_estimators,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,reg_lambda,objetivo_reg
200.0,0.02,255.0,100.0,0.8,0.6,5.0,null
400.0,0.05,127.0,50.0,0.8,0.8,1.0,regression_l1


## 3. Validación de origen móvil con la mejor configuración

En cada versión de variables, tarea y bloque: se entrena con todo lo anterior al bloque, se predice el bloque y se guardan las predicciones. Con las variables internas se mide además la importancia por permutación (insumo de 0420).

In [0]:
pred, imp = [], []
t_ini = time.time()
for version, numericas in VERSIONES.items():
    for tarea, y_col, _ in TAREAS:
        params = params_de(MEJORES, {MODELO: PARAMS_BASE}, version, tarea, MODELO)
        for ini, tr, va in PLIEGUES[tarea]:
            Dva, Xtr, ytr, Xva, yva = partes_del_bloque(df, HIST[ini], y_col, tr, va, CATEGORICAS + numericas)
            m = construir_modelo(tarea, MODELO, numericas, CATEGORICAS, params, SEMILLA).fit(Xtr, ytr)
            pred.append(marco_predicciones(version, tarea, ini, Dva, yva).assign(modelo=MODELO, pred=predecir(m, Xva, tarea)))
            if version == "internas":
                etiquetas = {"tarea": tarea, "modelo": MODELO, "bloque": ini.date()}
                imp.append(importancia_pliegue(m, Xva, yva, tarea, N_PERMUTACION, SEMILLA, etiquetas))
            print(f"{version} · {tarea} · {ini.date()} listo ({time.time() - t_ini:.0f} s)")
pred, imp = pd.concat(pred, ignore_index=True), pd.concat(imp, ignore_index=True)

internas · clf · 2026-04-01 listo (52 s)
internas · clf · 2026-05-01 listo (108 s)
internas · clf · 2026-06-01 listo (174 s)
internas · reg · 2026-04-01 listo (244 s)
internas · reg · 2026-05-01 listo (313 s)
internas · reg · 2026-06-01 listo (384 s)
internas+clima · clf · 2026-04-01 listo (431 s)
internas+clima · clf · 2026-05-01 listo (481 s)
internas+clima · clf · 2026-06-01 listo (549 s)
internas+clima · reg · 2026-04-01 listo (600 s)
internas+clima · reg · 2026-05-01 listo (660 s)
internas+clima · reg · 2026-06-01 listo (740 s)


## 4. Vista rápida de LightGBM

Solo para revisar este modelo. La comparación con los demás se hace en 0416.

In [0]:
display(tabla_metricas(pred, ["version", "tarea", "modelo"]).round(4))
display(tabla_metricas(pred, ["version", "tarea", "modelo", "bloque"]).round(4))

version,tarea,modelo,n,prevalencia,AP,AUC_ROC,Brier,LogLoss,MAE_h,RMSE_h,P90_error_abs_h,MAE_10pct_mas_lento_h,MAPE_pct_y>=1h,cobertura_MAPE
internas,clf,LightGBM,587957,0.0709,0.3471,0.7224,0.0563,0.2318,null,null,null,null,null,null
internas,reg,LightGBM,587951,null,null,null,null,null,6.9322,37.4037,7.3727,46.0116,64.2343,0.9655
internas+clima,clf,LightGBM,587957,0.0709,0.3449,0.7214,0.0562,0.2297,null,null,null,null,null,null
internas+clima,reg,LightGBM,587951,null,null,null,null,null,6.9981,37.3733,7.4003,45.7031,69.2281,0.9655


version,tarea,modelo,bloque,n,prevalencia,AP,AUC_ROC,Brier,LogLoss,MAE_h,RMSE_h,P90_error_abs_h,MAE_10pct_mas_lento_h,MAPE_pct_y>=1h,cobertura_MAPE
internas,clf,LightGBM,2026-04-01,196990,0.064,0.3543,0.7134,0.05,0.224,null,null,null,null,null,null
internas,clf,LightGBM,2026-05-01,205304,0.0742,0.3594,0.7316,0.0598,0.2461,null,null,null,null,null,null
internas,clf,LightGBM,2026-06-01,185663,0.0746,0.3368,0.7532,0.0591,0.2243,null,null,null,null,null,null
internas,reg,LightGBM,2026-04-01,196989,null,null,null,null,null,6.4355,37.541,6.915,38.3035,79.1812,0.967
internas,reg,LightGBM,2026-05-01,205301,null,null,null,null,null,6.7444,34.7021,7.8401,45.0933,57.0215,0.9646
internas,reg,LightGBM,2026-06-01,185661,null,null,null,null,null,7.6669,40.044,7.6023,55.0959,56.3137,0.965
internas+clima,clf,LightGBM,2026-04-01,196990,0.064,0.3551,0.7209,0.0497,0.2153,null,null,null,null,null,null
internas+clima,clf,LightGBM,2026-05-01,205304,0.0742,0.3457,0.7184,0.0596,0.2483,null,null,null,null,null,null
internas+clima,clf,LightGBM,2026-06-01,185663,0.0746,0.3361,0.7554,0.0593,0.2245,null,null,null,null,null,null
internas+clima,reg,LightGBM,2026-04-01,196989,null,null,null,null,null,6.5136,37.5812,7.0068,38.6589,81.4152,0.967


In [0]:
# Variables con más peso en validación (promedio de los tres bloques)
display(imp.groupby(["tarea", "variable"]).importancia.mean().sort_values(ascending=False).groupby(level=0).head(8).round(4))

tarea  variable            
reg    hist_h_zona_franja      0.7201
       hist_inc_zona_franja    0.3667
       categoria_zona          0.3282
       margen_sla_en_t0_h      0.3049
       n_hist                  0.2936
       hist_h_cliente          0.2699
       dia_semana              0.1445
clf    margen_sla_en_t0_h      0.1377
reg    n_hist_cliente          0.1158
clf    hist_inc_zona_franja    0.0823
       dia_semana              0.0682
       categoria_zona          0.0453
       ID_PRODUCTO             0.0175
       hist_h_zona_franja      0.0140
       n_hist                  0.0128
       ID_TIPOGUIA             0.0092
Name: importancia, dtype: float64

## 5. Guardar para 0416

In [0]:
pred.to_parquet(f"{CARPETA_MODELADO}pred_val_{MODELO}.parquet", index=False)
imp.to_parquet(f"{CARPETA_MODELADO}importancia_{MODELO}.parquet", index=False)
with open(RUTA_PARAMS, "w") as f:
    json.dump({"modelo": MODELO, "espacio": ESPACIO, "params_base": PARAMS_BASE, "semilla": SEMILLA,
               "n_configs": N_CONFIGS_BUSQUEDA, "max_filas": MAX_FILAS_BUSQUEDA, "mejores": MEJORES}, f, indent=2, default=str)
print("Guardado en", CARPETA_MODELADO)

Guardado en /Volumes/proyectogrado/source/source/LOG_CONSOLIDADO/REVISION_2026_10/V3/MODELADO/
